### 1 — Importar bibliotecas e validar o ambiente

In [ ]:
import os
import sys
from pathlib import Path
from importlib.metadata import version

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Selecione o interpretador /usr/bin/python3.11 "
        "no seletor de kernel do notebook."
    )

if not Path("/projects/python-packages/setup-completo.json").exists():
    raise RuntimeError(
        "A preparação do workspace não foi concluída. "
        "Consulte /projects/workspace-bootstrap.log."
    )

from kubeflow.spark import SparkClient
from pyspark.sql import functions as F
import pyarrow
import matplotlib.pyplot as plt

print("Python:", sys.executable)
print("Versão:", sys.version.split()[0])
print("Kubeflow SDK:", version("kubeflow"))
print("PySpark:", version("pyspark"))
print("PyArrow:", pyarrow.__version__)

### 2 - Configurar o processamento

In [ ]:
SPARK_REMOTE = os.environ.get("SPARK_REMOTE")

if not SPARK_REMOTE:
    raise RuntimeError("A variável SPARK_REMOTE não está configurada.")

NUM_TRANSACOES = 100_000
NUM_CLIENTES = 1_000
LIMITE_SUSPEITA = 9_500

assert NUM_TRANSACOES > 0
assert NUM_CLIENTES > 0

print("Endpoint:", SPARK_REMOTE)
print("Transações:", NUM_TRANSACOES)
print("Clientes:", NUM_CLIENTES)

### 3 — Conectar ao Spark existente

In [ ]:
import socket
from urllib.parse import urlparse

endpoint = urlparse(SPARK_REMOTE)

# Verifica a porta antes de iniciar a comunicação com o Spark.
with socket.create_connection(
    (endpoint.hostname, endpoint.port or 15002),
    timeout=5,
):
    print("Conexão TCP: OK")

if globals().get("spark") is None:
    client = SparkClient()
    spark = client.connect(base_url=SPARK_REMOTE)
    print("Sessão do notebook conectada.")
else:
    print("Reutilizando a sessão deste notebook.")

print("Spark remoto:", spark.version)
print("Registros de teste:", spark.range(1_000).count())

### 4 — Gerar transações fictícias

In [ ]:
transacoes = (
    spark.range(NUM_TRANSACOES)
    .withColumnRenamed("id", "transacao_id")
    .withColumn(
        "cliente_id",
        (F.col("transacao_id") % NUM_CLIENTES).cast("int"),
    )
    .withColumn(
        "valor",
        F.round(F.rand(seed=42) * 10_000 + 10, 2),
    )
    .withColumn(
        "canal",
        F.when(
            F.col("transacao_id") % 3 == 0,
            "mobile",
        ).otherwise("internet"),
    )
    .withColumn(
        "suspeita",
        (F.col("valor") > LIMITE_SUSPEITA).cast("int"),
    )
)

transacoes.printSchema()
transacoes.show(10, truncate=False)

total = transacoes.count()
assert total == NUM_TRANSACOES

print(f"Total de transações: {total:,}")

### 5 — Resumo por canal

In [ ]:
resumo = (
    transacoes.groupBy("canal")
    .agg(
        F.count("*").alias("total_transacoes"),
        F.round(F.sum("valor"), 2).alias("valor_total"),
        F.round(F.avg("valor"), 2).alias("ticket_medio"),
        F.sum("suspeita").alias("total_suspeitas"),
    )
    .withColumn(
        "percentual_suspeitas",
        F.round(
            F.col("total_suspeitas")
            / F.col("total_transacoes") * 100,
            2,
        ),
    )
)

resumo.orderBy("canal").show(truncate=False)

### 6 — Features por cliente

In [ ]:
features = transacoes.groupBy("cliente_id").agg(
    F.count("*").alias("quantidade_transacoes"),
    F.round(F.sum("valor"), 2).alias("valor_total"),
    F.round(F.avg("valor"), 2).alias("valor_medio"),
    F.round(F.max("valor"), 2).alias("maior_transacao"),
    F.sum("suspeita").alias("quantidade_suspeitas"),
)

print("TOP 10 CLIENTES POR VALOR MOVIMENTADO")

features.orderBy(
    F.desc("valor_total"),
    F.asc("cliente_id"),
).show(10, truncate=False)

### 7 — Consultar usando SQL

In [ ]:
transacoes.createOrReplaceTempView("transacoes_ficticias")

consulta = spark.sql("""
    SELECT
        canal,
        COUNT(*) AS quantidade,
        ROUND(AVG(valor), 2) AS valor_medio,
        ROUND(MAX(valor), 2) AS maior_valor
    FROM transacoes_ficticias
    WHERE suspeita = 1
    GROUP BY canal
    ORDER BY canal
""")

consulta.show(truncate=False)

### 8 — Visualizar os resultados

In [ ]:
resumo_local = resumo.orderBy("canal").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

resumo_local.plot.bar(
    x="canal",
    y="total_transacoes",
    ax=axes[0],
    legend=False,
    rot=0,
)

axes[0].set_title("Transações por canal")
axes[0].set_xlabel("Canal")
axes[0].set_ylabel("Quantidade")

resumo_local.plot.bar(
    x="canal",
    y="percentual_suspeitas",
    ax=axes[1],
    legend=False,
    rot=0,
    color="darkorange",
)

axes[1].set_title("Transações suspeitas por canal")
axes[1].set_xlabel("Canal")
axes[1].set_ylabel("Percentual (%)")

plt.tight_layout()
plt.show()

### 9 — Encerrar a sessão

In [ ]:
if globals().get("spark") is not None:
    spark.stop()
    spark = None

print("Sessão do notebook encerrada.")
print("O servidor Spark Connect permanece disponível.")